***
TITLE
***

Configuration

In [5]:
export=False


from pathlib import Path
import pandas as pd
import numpy as np
import time
import sys

PATH_GIT = Path.home() / 'Documents' / 'Github Repos' / 'Regional-Monitoring' / 'Indicator_Gen'
PATH_CODE    = PATH_GIT / 'Data' / 'RITIS'
PATH_CONFIG0 = PATH_GIT / 'config'
PATH_CONFIG  = PATH_CODE / 'config'
PATH_SQL = PATH_GIT / 'Data' / 'RITIS' / 'sql_scripts'#not needed because I wont be running the sql scripts in this script

sys.path.append(str(PATH_CONFIG0))
import functions as func

pd.set_option('display.max_columns', None)

# I Drive Path to monthly csv files
PATH_IDRIVE = Path(r"I:/Projects/Josh/Regional Monitoring/Congestion/monthly csv")
# SharePoint
PATH_SP = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' 
PATH_CONGESTION = PATH_SP / 'Data' / 'Safe Equitable Resilient Infrastructure' / 'Congestion'
PATH_PHED  = PATH_CONGESTION / 'RITIS' / 'PHED'
PATH_LOTTR = PATH_CONGESTION / 'RITIS' / 'LOTTR'
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")

In [9]:
# Option 1: Use parentheses
df_jan2024 = pd.read_csv(PATH_IDRIVE / '2024_01 (1)' / '2024_01.csv')



In [11]:
df_jan2024.head()

,tmc_code,measurement_tstamp,speed,historical_average_speed,reference_speed,travel_time_seconds,data_density,NPMRDS2 2024
0,105P17071,2024-01-01 00:00:00,NaN,NaN,34.0,NaN,NaN,NaN
1,105P17071,2024-01-01 00:15:00,NaN,NaN,34.0,NaN,NaN,NaN
2,105P17071,2024-01-01 00:30:00,NaN,NaN,34.0,NaN,NaN,NaN
3,105P17071,2024-01-01 00:45:00,NaN,NaN,34.0,NaN,NaN,NaN
4,105P17071,2024-01-01 01:00:00,NaN,NaN,34.0,NaN,NaN,NaN


In [ ]:
# Free-flow period parameters
FF_PERIOD_START = 20  # Free-flow period starts at or after this hour (8 PM)
FF_PERIOD_END = 6     # Free-flow period ends before this hour (6 AM)

# Weekdays
WEEKDAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

***
Functions
***

In [ ]:
def load_monthly_data(csv_path):
    """
    Load monthly NPMRDS data from CSV file.
    Expected columns: tmc_code, measurement_tstamp, speed, travel_time_seconds
    """
    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    df['measurement_tstamp'] = pd.to_datetime(df['measurement_tstamp'])
    return df

In [ ]:
def load_tmc_metadata(tmc_metadata_path):
    """
    Load TMC metadata (road characteristics).
    Expected columns: tmc, road, route_numb, f_system, nhs, miles
    """
    print(f"Loading TMC metadata from {tmc_metadata_path}...")
    tmc_meta = pd.read_csv(tmc_metadata_path)
    return tmc_meta

In [ ]:
def calculate_freeflow_speed(traffic_data, tmc_metadata):
    """
    Calculate free-flow speed for each TMC based on overnight periods.
    - Freeways (f_system 1,2): 85th percentile
    - Arterials: 60th percentile
    """
    print("Calculating free-flow speeds...")
    
    # Filter for overnight free-flow period
    ff_data = traffic_data.copy()
    ff_data['hour'] = ff_data['measurement_tstamp'].dt.hour
    ff_mask = (ff_data['hour'] >= FF_PERIOD_

In [ ]:
def calculate_freeflow_speed(traffic_data, tmc_metadata):
    """
    Calculate free-flow speed for each TMC based on overnight periods.
    - Freeways (f_system 1,2): 85th percentile
    - Arterials: 60th percentile
    """
    print("Calculating free-flow speeds...")
    
    # Filter for overnight free-flow period
    ff_data = traffic_data.copy()
    ff_data['hour'] = ff_data['measurement_tstamp'].dt.hour
    ff_mask = (ff_data['hour'] >= FF_PERIOD_START) | (ff_data['hour'] < FF_PERIOD_END)
    ff_data = ff_data[ff_mask]
    
    # Merge with TMC metadata to get f_system
    ff_data = ff_data.merge(tmc_metadata[['tmc', 'f_system', 'nhs']], 
                            left_on='tmc_code', right_on='tmc', how='left')
    
    # Filter for NHS roads only
    ff_data = ff_data[ff_data['nhs'] > 0]
    
    # Calculate percentile based on facility type
    def calc_percentile(group):
        f_system = group['f_system'].iloc[0]
        if f_system in [1, 2]:  # Freeway
            return group['speed'].quantile(0.85)
        else:  # Arterial
            return group['speed'].quantile(0.60)
    
    ff_speeds = ff_data.groupby('tmc_code').apply(calc_percentile).reset_index()
    ff_speeds.columns = ['tmc_code', 'ff_speed_art60thp']
    
    # Count overnight epochs
    epochs_night = ff_data.groupby('tmc_code').size().reset_index(name='epochs_night')
    
    return ff_speeds, epochs_night

In [ ]:
def calculate_hourly_speeds(traffic_data, ff_speeds):
    """
    Calculate average speeds by hour of day for weekdays.
    """
    print("Calculating hourly congestion patterns...")
    
    # Filter for weekdays only
    weekday_data = traffic_data.copy()
    weekday_data['day_name'] = weekday_data['measurement_tstamp'].dt.day_name()
    weekday_data = weekday_data[weekday_data['day_name'].isin(WEEKDAYS)]
    weekday_data['hour'] = weekday_data['measurement_tstamp'].dt.hour
    
    # Merge with free-flow speeds
    weekday_data = weekday_data.merge(ff_speeds, on='tmc_code', how='inner')
    
    # Calculate harmonic average speed by TMC and hour
    hourly_stats = weekday_data.groupby(['tmc_code', 'hour']).agg({
        'speed': lambda x: len(x) / (1.0 / x).sum(),  # Harmonic mean
        'travel_time_seconds': 'mean',
        'measurement_tstamp': 'count',
        'ff_speed_art60thp': 'first'
    }).reset_index()
    
    hourly_stats.columns = ['tmc_code', 'hour_of_day', 'havg_spd_weekdy', 
                            'avg_tt_sec_weekdy', 'total_epochs_hr', 'ff_speed_art60thp']
    
    # Filter out hours with insufficient data
    hourly_stats = hourly_stats[hourly_stats['total_epochs_hr'] >= 100]
    
    # Calculate congestion ratio
    hourly_stats['cong_ratio_hr_weekdy'] = (hourly_stats['havg_spd_weekdy'] / 
                                             hourly_stats['ff_speed_art60thp'])
    
    # Rank hours by congestion (lowest ratio = most congested)
    hourly_stats['hour_cong_rank'] = (hourly_stats.groupby('tmc_code')['cong_ratio_hr_weekdy']
                                      .rank(method='first', ascending=True))
    
    return hourly_stats

In [ ]:
def calculate_worst_hours_speed(traffic_data, ff_speeds, hourly_stats):
    """
    Calculate harmonic average speed during the worst 4 hours.
    """
    print("Calculating worst 4 hours congestion...")
    
    # Get worst 4 hours for each TMC
    worst_hours = hourly_stats[hourly_stats['hour_cong_rank'] < 5][['tmc_code', 'hour_of_day']]
    
    # Filter traffic data for weekdays
    weekday_data = traffic_data.copy()
    weekday_data['day_name'] = weekday_data['measurement_tstamp'].dt.day_name()
    weekday_data = weekday_data[weekday_data['day_name'].isin(WEEKDAYS)]
    weekday_data['hour'] = weekday_data['measurement_tstamp'].dt.hour
    
    # Merge to get only worst hours
    worst_data = weekday_data.merge(worst_hours, 
                                     left_on=['tmc_code', 'hour'], 
                                     right_on=['tmc_code', 'hour_of_day'], 
                                     how='inner')
    
    # Merge with free-flow speeds
    worst_data = worst_data.merge(ff_speeds, on='tmc_code', how='inner')
    
    # Calculate harmonic average for worst 4 hours
    worst_stats = worst_data.groupby('tmc_code').agg({
        'speed': lambda x: len(x) / (1.0 / x).sum(),  # Harmonic mean
        'measurement_tstamp': 'count',
        'ff_speed_art60thp': 'first'
    }).reset_index()
    
    worst_stats.columns = ['tmc_code', 'havg_spd_worst4hrs', 'epochs_worst4hrs', 'ff_speed_art60thp']
    
    return worst_stats[['tmc_code', 'havg_spd_worst4hrs', 'epochs_worst4hrs']]

In [ ]:
def calculate_slowest_hour(hourly_stats):
    """
    Find the single slowest hour for each TMC.
    """
    print("Identifying slowest hour...")
    
    slowest = hourly_stats[hourly_stats['hour_cong_rank'] == 1].copy()
    slowest = slowest[['tmc_code', 'hour_of_day', 'havg_spd_weekdy', 'total_epochs_hr']]
    slowest.columns = ['tmc_code', 'slowest_hr', 'slowest_hr_speed', 'epochs_slowest_hr']
    
    # Remove duplicates (keep first if multiple hours tied)
    slowest = slowest.drop_duplicates(subset=['tmc_code'], keep='first')
    
    return slowest

In [ ]:
def create_final_report(tmc_metadata, ff_speeds, worst_stats, slowest_stats, epochs_night):
    """
    Combine all metrics into final report.
    """
    print("Creating final report...")
    
    # Start with TMC metadata for NHS roads
    final = tmc_metadata[tmc_metadata['nhs'] > 0].copy()
    
    # Merge all calculated metrics
    final = final.merge(ff_speeds, left_on='tmc', right_on='tmc_code', how='left')
    final = final.merge(worst_stats, left_on='tmc', right_on='tmc_code', how='left')
    final = final.merge(slowest_stats, left_on='tmc', right_on='tmc_code', how='left')
    final = final.merge(epochs_night, left_on='tmc', right_on='tmc_code', how='left')
    
    # Fill missing values with -1
    metric_cols = ['ff_speed_art60thp', 'havg_spd_worst4hrs', 'slowest_hr', 
                   'slowest_hr_speed', 'epochs_worst4hrs', 'epochs_slowest_hr', 'epochs_night']
    for col in metric_cols:
        final[col] = final[col].fillna(-1.0)
    
    # Calculate congestion ratios
    final['congratio_worst4hrs'] = np.where(
        (final['havg_spd_worst4hrs'] > -1) & (final['ff_speed_art60thp'] > -1),
        np.minimum(final['havg_spd_worst4hrs'] / final['ff_speed_art60thp'], 1.0),
        -1.0
    )
    
    final['congratio_worsthr'] = np.where(
        (final['slowest_hr_speed'] > -1) & (final['ff_speed_art60thp'] > -1),
        final['slowest_hr_speed'] / final['ff_speed_art60thp'],
        -1.0
    )
    
    # Select and order columns
    output_cols = ['tmc', 'road', 'route_numb', 'f_system', 'nhs', 'miles',
                   'ff_speed_art60thp', 'havg_spd_worst4hrs', 'congratio_worst4hrs',
                   'slowest_hr', 'slowest_hr_speed', 'congratio_worsthr',
                   'epochs_worst4hrs', 'epochs_slowest_hr', 'epochs_night']
    
    final = final[output_cols]
    
    return final

In [ ]:
def calculate_system_metrics(final_report):
    """
    Calculate system-wide congestion metrics.
    """
    print("\nCalculating system-wide metrics...")
    
    # Filter valid data
    valid_data = final_report[
        (final_report['havg_spd_worst4hrs'] > -1) & 
        (final_report['ff_speed_art60thp'] > -1)
    ]
    
    tot_nhs_dirmiles = valid_data['miles'].sum()
    
    # Miles where congestion < 60% of free-flow
    congested_miles = valid_data[
        valid_data['havg_spd_worst4hrs'] / valid_data['ff_speed_art60thp'] < 0.6
    ]['miles'].sum()
    
    pct_dirmi_congested = (congested_miles / tot_nhs_dirmiles * 100) if tot_nhs_dirmiles > 0 else 0
    
    print(f"\nTotal NHS directional miles: {tot_nhs_dirmiles:,.2f}")
    print(f"Congested miles (<60% free-flow): {congested_miles:,.2f}")
    print(f"Percent of miles congested: {pct_dirmi_congested:.2f}%")
    
    return {
        'tot_nhs_dirmiles': tot_nhs_dirmiles,
        'congested_miles': congested_miles,
        'pct_dirmi_congested': pct_dirmi_congested
    }

*** 
Execution
***

In [ ]:
def analyze_congestion(traffic_csv_path, tmc_metadata_path, output_path=None):
    """
    Main function to run the complete congestion analysis.
    
    Parameters:
    -----------
    traffic_csv_path : str
        Path to monthly traffic data CSV file
    tmc_metadata_path : str
        Path to TMC metadata CSV file
    output_path : str, optional
        Path to save the final report CSV
    """
    
    # Load data
    traffic_data = load_monthly_data(traffic_csv_path)
    tmc_metadata = load_tmc_metadata(tmc_metadata_path)
    
    # Calculate metrics
    ff_speeds, epochs_night = calculate_freeflow_speed(traffic_data, tmc_metadata)
    hourly_stats = calculate_hourly_speeds(traffic_data, ff_speeds)
    worst_stats = calculate_worst_hours_speed(traffic_data, ff_speeds, hourly_stats)
    slowest_stats = calculate_slowest_hour(hourly_stats)
    
    # Create final report
    final_report = create_final_report(tmc_metadata, ff_speeds, worst_stats, 
                                       slowest_stats, epochs_night)
    
    # Calculate system metrics
    system_metrics = calculate_system_metrics(final_report)
    
    # Save output if path provided
    if output_path:
        final_report.to_csv(output_path, index=False)
        print(f"\nFinal report saved to: {output_path}")
    
    return final_report, system_metrics

*** 
Example
***

In [ ]:
if __name__ == "__main__":
    # Example: Analyze a single month
    traffic_csv = "npmrds_2024_01_paxtruck_comb.csv"  # Change to your file
    tmc_meta_csv = "npmrds_2024_alltmc_txt.csv"       # Change to your file
    output_csv = "congestion_report_2024_01.csv"
    
    try:
        final_report, metrics = analyze_congestion(traffic_csv, tmc_meta_csv, output_csv)
        print("\nAnalysis complete!")
        print(f"\nProcessed {len(final_report)} TMCs")
        
    except FileNotFoundError as e:
        print(f"Error: Could not find file - {e}")
    except Exception as e:
        print(f"Error during analysis: {e}")